# Introdução à Inteligência Artificial 2025/2026

## Projeto 4: Lagartas Satisfeitas

### Entrega: 5 de Dezembro às 23:59

***

<left> <img src="figures/lagartas_P4.png" width="400" /> </center>

## Introdução e Objetivos

Neste projeto voltamos ao mundo da lagarta, desta vez com várias lagartas e várias maçãs, em que cada lagarta deve comer uma (e só uma) maçã. Um único agente controla todas as lagartas; cada ação move apenas uma lagarta numa determinada direção. Vamos ajudar a procura de soluções descobrindo estados em que não é possível cada lagarta comer uma maçã, e assim evitar andar à procura de uma solução que não existe. Faremos isto através de modelização e resolução de Problemas de Satisfação de Restrições (CSPs - *Constraint Satisfaction Problems*).

Vão precisar dos módulos seguintes (distribuídos juntamente com o enunciado):
* `lagarta.py`
* `search.py`
* `searchPlus_better.py`
* `csp_v3.py`
* `utils.py`


#### Recordar o Mundo da Lagarta e a Procura da Lagarta (1 lagarta, 1 maçã)

Podem recordar o Mundo da Lagarta (projeto 1) e a Procura da Lagarta (projeto 2) nos enunciados respetivos, mas fica aqui uma breve descrição:

O Mundo da Lagarta original é um mundo simples onde existem apenas paredes e uma maçã, para além da própria lagarta. As paredes rodeiam completamente o mundo (que pode ser quadrado ou retangular) e podem existir também no seu interior. A maçã pode estar num sítio qualquer, apoiada em cima de uma parede ou a levitar, mas a lagarta está sujeita à força de gravidade:

* a posição da lagarta é definida pela posição da sua cabeça;
* a lagarta só pode mover-se ortogonalmente (não na diagonal) para casas imediatamente adjacentes à casa onde está posicionada;
* a cada movimento que faz, a lagarta cresce: a cabeça move-se mas fica corpo onde estava a cabeça;
* a lagarta só pode mover-se para casas livres (não pode atravessar paredes nem o seu próprio corpo) ou para a casa onde está a maçã;
* se a cabeça da lagarta não estiver apoiada (a célula imediatamente abaixo está livre) a única ação possível é para baixo (B);
* a ação de levantar a cabeça para cima (C) só é possível se o nível máximo de esforço ainda não tiver sido atingido (nível máximo = 3), e esta ação aumenta o nível de esforço em 1 unidade;
* se a lagarta estiver em esforço (nível de esforço > 0) só pode ir para a direita (D) ou esquerda (E) se isso a colocar numa casa com apoio;
* independentemente do nível de esforço anterior, se a lagarta se move para uma casa com apoio o seu nível de esforço passa a ser 0;
* uma casa com apoio é uma casa imediatamente acima de uma parede ou do corpo da lagarta.

Na Procura da Lagarta, as ações da lagarta têm custos diferenciados. Mais precisamente:

* ir para baixo tem custo 1;
* ir para a esquerda ou para a direita tem custo 2;
* ir para cima tem custo 3.

O objetivo da lagarta é chegar à maçã com o menor custo possível.

#### O que mudou (*n* lagartas, *n* maçãs)

Neste projeto, o mundo da lagarta pode agora conter várias lagartas e várias maçãs, sempre em igual número. O objetivo é que cada lagarta consiga comer uma maçã. Sendo assim:

* as posições das ***n* lagartas** são definidas pelas posições das *n* cabeças; também existem agora ***n* maçãs**;
* cada lagarta só pode mover-se para casas livres (não pode atravessar paredes nem corpo, **nem a cabeça de outra lagarta**) ou para uma casa onde está uma maçã;
* as regras para determinar as ações possíveis mantêm-se inalteradas, mas agora uma casa com apoio pode ser também uma casa **imediatamente acima da cabeça de *outra* lagarta**.


#### `lagarta.py`

Todo o código que implementa as descrições acima é fornecido em `lagarta.py` e **não deve ser alterado**. Deve, no entanto, ser consultado para verem como o problema está representado. Em particular, reparem que agora o estado guarda a localização de cada (cabeça de) lagarta e de cada maçã, o esforço de cada lagarta, e a localização de todas as casas que têm corpo de lagarta sem distinguir a que lagarta(s) pertence o corpo.

Cada ação só move uma lagarta; os custos dos movimentos são os mesmos para todas as lagartas; o custo total é a soma dos custos de todas as ações, logo, dos movimentos de todas as lagartas; as heurísticas fornecidas somam as distâncias (h_dist) eventualmente ponderadas pelos custos (h_dist_costs) entre todas as lagartas e todas as maçãs (não sendo admissíveis, mas não se preocupem com isso).


## Procura mais eficiente com CSP

A procura por soluções é por vezes extremamente ineficiente, pois na árvore de procura atingem-se estados a partir dos quais já não é possível encontrar uma solução para o problema e no entanto a procura continua nesses ramos enquanto existirem ações possíveis para explorar. Dois exemplos de estados "sem saída" são mostrados em baixo. No primeiro estado (esquerda), uma das maçãs está bloqueada por paredes e por isso é impossível chegar-lhe; no segundo estado (direita), uma passagem estreita faz com que apenas uma das lagartas consiga chegar a uma maçã, sendo que a segunda lagarta tem o caminho bloqueado pelo corpo da primeira.

```
= = = = = = =     = = = = = = =
= x = . x . =     = x . . . x =
= = . . = . =     = = = . = = =
= . . . . . =     = . . @ . . =
= @ . . . @ =     = o o o . @ =
= = = = = = =     = = = = = = =
```

Vamos conseguir reconhecer alguns destes estados "sem saída" através de modelização e resolução de Problemas de Satisfação de Restrições (CSPs - *Constraint Satisfaction Problems*) e assim tornar as procuras mais eficientes pois iremos impedir a exploração de ramos de procura que não levam a soluções. Os alunos devem implementar três funções, descritas no resto do enunciado:

* `csp_possivel_solucao`
* `csp_encontra_alcancaveis_1maca`
* `encontra_alcancaveis_todas_macas`

## Função `csp_possivel_solucao`
 
Em primeiro lugar, vamos usar uma função fornecida por nós, chamada `possivel_solucao`, que recebe como input

1. As lagartas (localizações das suas cabeças);
2. Uma tabela formada pelos pares (célula,maçãs) em que cada entrada da tabela (dicionário) indica quais as maçãs alcançáveis (as suas coordenadas) por uma lagarta que esteja na célula. Excluem-se as células em que não pode estar uma lagarta (paredes e corpo).

e devolve uma possível atribuição de lagartas a maçãs, ou None caso seja impossível cada lagarta atingir uma maçã (estado "sem saída"). Esta função não garante detetar todos os casos em que é impossível chegar a uma solução, mas todos os que deteta são efetivamente estados sem saída. A função é integrada no método `actions` do problema de procura para que, quando é detetada a impossibilidade de atingir uma solução, não seja devolvida qualquer ação possível.

Vejamos um exemplo que ilustra as maçãs alcançáveis:

<img src="figures/exemplo_possivel_solucao.png" alt="Drawing" style="width: 400px;"/>

Na imagem da esquerda, as maçãs estão identificadas com os números 1 e 2. Na imagem da direita vê-se a atribuição, a cada célula onde pode estar uma (cabeça de) lagarta, de uma lista de maçãs que podem ser alcançadas a partir dessa célula.

A função `possivel_solucao` será como especificada abaixo, e vai usar a **vossa** função **`csp_possivel_solucao`** da seguinte forma:

In [ ]:
from lagarta import *

def csp_possivel_solucao(lagartas,alcancaveis):
    pass
    return csp_lagartas1


def possivel_solucao(lagartas,alcancaveis):
    csp_lagartas1 = csp_possivel_solucao(lagartas,alcancaveis) # <--- a vossa função csp_possivel_solucao 
    r = backtracking_search(csp_lagartas1, inference = forward_checking)
    return r

Para implementarem a vossa função **`csp_possivel_solucao`**, devem formular o problema como um CSP. A função recebe como input exatamente o mesmo que a função `possivel_solucao` e deve retornar como output uma instância de um problema CSP. Notem que o problema pode ser visto como um problema de coloração de mapas em que:

- As células com (cabeça de) lagarta são as variáveis (as regiões num mapa);
- Os domínios são dados pela lista de maçãs alcançáveis pelas respectivas células (as cores para colorir o mapa);
- Duas células são vizinhas se partilharem alguma das maçãs (se forem regiões contíguas);
- Duas células vizinhas não podem ter o mesmo valor (duas regiões contíguas não podem ter a mesma cor).

Lembrem-se que ambos os inputs são dados, i.e., as células com lagartas e a atribuição de células a listas de maçãs (as maçãs alcançáveis).

Iremos testar se as vossas variáveis, domínios, vizinhos e restrições estão corretas, da forma exemplificada em baixo. Recordamos que

-  =  é uma parede
-  @  é uma cabeça de lagarta
-  o  é corpo de lagarta
-  x  é uma maçã
-  .  é uma casa vazia

e que as coordenadas das casas (usadas em `alcancaveis`) começam em (0,0) no canto inferior esquerdo. Para recordarem mais coisas podem recorrer aos enunciados dos projetos 1 e/ou 2.

In [ ]:
from lagarta import *

line1 = "= = = = = = =\n"
line2 = "= . . . . . =\n"
line3 = "= . . = = x =\n"
line4 = "= @ . . . = =\n"
line5 = "= = . = = . =\n"
line6 = "= . . @ x . =\n"
line7 = "= = = = = = =\n"
grelha = line1 + line2 + line3 + line4 + line5 + line6 + line7

alcancaveis={(1, 1): [(5, 4)], (1, 3): [(5, 4)], (1, 4): [(5, 4)], (1, 5): [(5, 4)], (2, 1): [(5, 4)], (2, 2): [(5, 4)], (2, 3): [(5, 4)], (2, 4): [(5, 4)], (2, 5): [(5, 4)], (3, 1): [(4, 1), (5, 4)], (3, 3): [(5, 4)], (3, 5): [(5, 4)], (4, 1): [(4, 1)], (4, 3): [(5, 4)], (4, 5): [(5, 4)], (5, 1): [(4, 1)], (5, 2): [(4, 1)], (5, 4): [(5, 4)], (5, 5): [(5, 4)]}
                   
l = MundoLagarta(grelha)
lagartas = l.initial['heads']
csp_lagarta1 = csp_possivel_solucao(lagartas,alcancaveis)

print('Variáveis:',sorted(csp_lagarta1.variables))
print('Domínios:',dict(sorted(csp_lagarta1.domains.items())))
sorted_neighbors = {key: sorted(values) for key, values in sorted(csp_lagarta1.neighbors.items())}
print('Vizinhos:',dict(sorted(sorted_neighbors.items())))
print('Restrição obedecida?',csp_lagarta1.constraints((1,3),(1,1),(3,1),(1,1)))
print('Restrição obedecida?',csp_lagarta1.constraints((1,3),(1,1),(3,1),(2,1)))

Claro que não vai funcionar se ainda não programaram a função `csp_possivel_solucao`. O resultado da célula anterior deve ser:

    Variáveis: [(1, 3), (3, 1)]
    Domínios: {(1, 3): [(5, 4)], (3, 1): [(4, 1), (5, 4)]}
    Vizinhos: {(1, 3): [(3, 1)], (3, 1): [(1, 3)]}
    Restrição obedecida? False
    Restrição obedecida? True

Ordenamos sempre os resultados para que vocês não tenham de se preocupar com isso nos testes automáticos.

Também iremos testar a vossa função assim:

In [ ]:
from lagarta import *

line1 = "= = = = = = =\n"
line2 = "= x @ . . . =\n"
line3 = "= = = = . x =\n"
line4 = "= @ . . . = =\n"
line5 = "= = = = = x =\n"
line6 = "= . . @ x @ =\n"
line7 = "= = = = = = =\n"
grelha = line1 + line2 + line3 + line4 + line5 + line6 + line7

alcancaveis={(1, 1): [], (1, 3): [(5, 4)], (1, 5): [(1, 5)], (2, 1): [], (2, 3): [(5, 4)], (2, 5): [(1, 5), (5, 4)], (3, 1): [(4, 1)], (3, 3): [(5, 4)], (3, 5): [(5, 4)], (4, 1): [(4, 1)], (4, 3): [(5, 4)], (4, 4): [(5, 4)], (4, 5): [(5, 4)], (5, 1): [(4, 1), (5, 2)], (5, 2): [(5, 2)], (5, 4): [(5, 4)], (5, 5): [(5, 4)]}
                   
l = MundoLagarta(grelha)
lagartas = l.initial['heads']
result = possivel_solucao(lagartas,alcancaveis) # <--- usa a vossa função csp_possivel_solucao
if result != None:
    result = dict(sorted(result.items()))
print(result)


Aqui o resultado devia ser:

    {(1, 3): (5, 4), (2, 5): (1, 5), (3, 1): (4, 1), (5, 1): (5, 2)}

Vejam um caso onde, apesar de ser muito parecido (apenas uma parede adicionada), não há solução possível:

In [ ]:
from lagarta import *

line1 = "= = = = = = =\n"
line2 = "= x @ . . . =\n"
line3 = "= = = = . x =\n"
line4 = "= @ . . = = =\n"
line5 = "= = . = = x =\n"
line6 = "= . . @ x @ =\n"
line7 = "= = = = = = =\n"
grelha = line1 + line2 + line3 + line4 + line5 + line6 + line7

alcancaveis={(1, 1): [], (1, 3): [], (1, 5): [(1, 5)], (2, 1): [], (2, 2): [], (2, 3): [], (2, 5): [(1, 5), (5, 4)], (3, 1): [(4, 1)], (3, 3): [], (3, 5): [(5, 4)], (4, 1): [(4, 1)], (4, 4): [(5, 4)], (4, 5): [(5, 4)], (5, 1): [(4, 1), (5, 2)], (5, 2): [(5, 2)], (5, 4): [(5, 4)], (5, 5): [(5, 4)]}
                   
l = MundoLagarta(grelha)
lagartas = l.initial['heads']
result = possivel_solucao(lagartas,alcancaveis) # <--- usa a vossa função csp_possivel_solucao
if result != None:
    result = dict(sorted(result.items()))
print(result)


O resultado da célula anterior deve ser:

    None

## Função `csp_encontra_alcancaveis_1maca`

No exemplo anterior, a variável `alcancaveis` era dada, mas queremos ser nós a obtê-la automaticamente com base no mundo. Vamos formular este problema também como um CSP. Na verdade, vamos ter um CSP por cada maçã, numa função chamada `encontra_alcancaveis_1maca` que recebe como input

1. Uma instância do mundo da lagarta;
2. As coordenadas de uma maçã desse problema.

e devolve, para cada célula onde possa estar uma lagarta, a informação sobre se uma lagarta nessa célula consegue chegar à maçã especificada (1 se sim, 0 se não).

A função `encontra_alcancaveis_1maca` será como especificada abaixo, e vai usar a **vossa** função **`csp_encontra_alcancaveis_1maca`** da seguinte forma:

In [ ]:
from lagarta import *

def csp_encontra_alcancaveis_1maca(s,goal):
    pass
    return csp_lagartas2


def encontra_alcancaveis_1maca(s,goal):
    csp_lagartas2 = csp_encontra_alcancaveis_1maca(s,goal) # <--- a vossa função csp_encontra_alcancaveis_1maca 
    r = backtracking_search(csp_lagartas2, order_domain_values = number_ascending_order, inference = forward_checking)    
    return {} if r == None else r

Para implementarem a função **`csp_encontra_alcancaveis_1maca`** devem formular o problema como um CSP em que:

- As células onde podem estar lagartas são as variáveis;
- Os domínios são a possibilidade (1) ou não (0) de uma lagarta partir da célula e chegar à maçã;
- Duas células são vizinhas se forem adjacentes (na vertical ou horizontal) e se for possível uma lagarta passar de uma para outra, num dos sentidos ou nos dois. Se forem adjacentes, mas não é possível a passagem em nenhum dos sentidos, então não são vizinhas. Para SIMPLIFICAR, não vamos olhar para os níveis de esforço das lagartas, e sendo assim não há muitas situações em que seja possível uma lagarta passar numa direção mas não na direção oposta;
- As restrições, tantos as unárias como as binárias, ficam por vossa conta!

Reparem que este CSP pode devolver soluções que, apesar de válidas no contexto do problema, não ajudam a procura tanto como outras. Por exemplo, se duas soluções válidas tiverem como única diferença o valor atribuído a uma das células (0 num caso, 1 no outro), a solução que mais ajuda a procura é aquela que tem o 0, pois permite mais facilmente detetar um estado "sem saída". É por isso que a função `find_alcancaveis_1maca` faz sempre a procura com `order_domain_values = number_ascending_order`, como especificado acima.

Iremos testar a vossa função desta forma:

In [ ]:
from lagarta import *

line1 = "= = = = = = =\n"
line2 = "= @ . . . = =\n"
line3 = "= = = = = . =\n"
line4 = "= . . . x . =\n"
line5 = "= = = = = = =\n"
grelha = line1 + line2 + line3 + line4 + line5

l = MundoLagarta(grelha)
alcancaveis = encontra_alcancaveis_1maca(l,(4,1))  # <--- usa a vossa função csp_encontra_alcancaveis_1maca
alcancaveis = dict(sorted(alcancaveis.items()))
print(alcancaveis)

Mais uma vez, só funciona quando completarem a função `csp_encontra_alcancaveis_1maca`. O resultado da célula anterior deve ser:

    {(1, 1): 1, (1, 3): 0, (2, 1): 1, (2, 3): 0, (3, 1): 1, (3, 3): 0, (4, 1): 1, (4, 3): 0, (5, 1): 1, (5, 2): 1}

## Função `encontra_alcancaveis_todas_macas`

No exemplo anterior o mundo da lagarta só tinha uma maçã, mas pode ter mais do que uma. Precisamos agora de uma função **vossa** chamada **`encontra_alcancaveis_todas_macas`** que chama a `encontra_alcancaveis_1maca` tantas vezes quantas o número de maçãs no mundo. Recebe uma instância do mundo da lagarta e devolve a variável `alcancaveis` que é construída a partir dos resultados das várias chamadas.

Mostramos em baixo a estrutura que aconselhamos para esta função. Ordenem as maçãs como mostrado, para que os resultados sejam exatamente como esperado.

In [ ]:
from lagarta import *

def encontra_alcancaveis_todas_macas(s):
    sorted_goals = sorted(list(s.goal))
    pass # o vosso código aqui
    return result_alcancaveis


Iremos testar a vossa função da seguinte forma:

In [ ]:
from lagarta import *

line1 = "= = = = = =\n"
line2 = "= = x = @ =\n"
line3 = "= x = = x =\n"
line4 = "= @ . . @ =\n"
line5 = "= = = = = =\n"
grelha = line1 + line2 + line3 + line4 + line5

l = MundoLagarta(grelha)
result = encontra_alcancaveis_todas_macas(l)
result = dict(sorted(result.items()))
print(result)

A célula anterior devia dar:

    {(1, 1): [(1, 2)], (1, 2): [(1, 2)], (2, 1): [], (2, 3): [(2, 3)], (3, 1): [], (4, 1): [(4, 2)], (4, 2): [(4, 2)], (4, 3): [(4, 2)]}

## Tudo junto nas Lagartas Satisfeitas

Implementadas as três funções **`csp_possivel_solucao`**, **`csp_encontra_alcancaveis_1maca`** e **`encontra_alcancaveis_todas_macas`**, podemos finalmente juntar tudo para realizar procuras mais eficientes. Usamos o algoritmo A* com a heurística `h_dist_costs` que se encontra no código fornecido e que não devem alterar. Vejam como, num exemplo tão simples de um problema que tem solução, mesmo assim no mundo das Lagartas Satisfeitas conseguimos reduzir o número de estados visitados, comparado com as lagartas que não têm ajuda para detetar estados sem saída.

In [ ]:
from lagarta import *

line1 = "= = = = = = =\n"
line2 = "= x @ . . . =\n"
line3 = "= = = = . x =\n"
line4 = "= @ . . . = =\n"
line5 = "= = = = = . =\n"
line6 = "= . . . . . =\n"
line7 = "= = = = = = =\n"
grelha = line1 + line2 + line3 + line4 + line5 + line6 + line7

l = MundoLagarta(grelha)
res, exp = astar_search_plus_count(l,l.h_dist_costs)
if res == None:
    print('No solution!')
    print('Explorados sem CSP:',exp)
else:
    print('Explorados sem CSP:',exp)
    print(res.solution())

l = MundoLagarta(grelha,plug_in_function=possivel_solucao)
l.alcancaveis = encontra_alcancaveis_todas_macas(l)
res, exp = astar_search_plus_count(l,l.h_dist_costs)
if res == None:
    print('No solution!')
    print('Explorados com CSP:',exp)
else:
    print('Explorados com CSP:',exp)
    print(res.solution())

O resultado da célula anterior deve ser:

    Explorados sem CSP: 16
    [(0, 'D'), (0, 'D'), (0, 'D'), (0, 'C'), (1, 'E'), (0, 'D')]
    Explorados com CSP: 14
    [(0, 'D'), (0, 'D'), (0, 'D'), (0, 'C'), (1, 'E'), (0, 'D')]

## Submissão

### Quiz
Cada aluno deve completar as implementações pedidas e testá-las no link do *quiz* **Avaliação Contínua 4** que está na página da disciplina, introduzindo aí o vosso código. Pode-se submeter e avaliar o código várias vezes, sendo a submissão com melhor nota a que será considerada. Este projeto deve ser realizado **individualmente**.

Esse *quiz* é constituído por uma única pergunta, onde as implementações das diferentes funções são avaliadas através de um conjunto de testes automáticos visíveis e mais alguns testes escondidos. Alguns testes são específicos para a função `csp_possivel_solucao` (valendo 10 valores), outros para a função `csp_encontra_alcancaveis_1maca` (valendo 5 valores). Outros testes avaliam o par de funções `csp_encontra_alcancaveis_1maca` e `encontra_alcancaveis_todas_macas` a trabalhar juntas (valendo 3 valores), e finalmente os últimos testes avaliam todas as funções a trabalhar em conjunto na procura A* (valendo 2 valores). Os testes visíveis valem 6 em 20 e os testes invisíveis valem 14 em 20. A nota obtida nesta avaliação será convertida da escala de 0 a 20 para a escala de 0 a 2.15.

### Ficheiro Pyhton
Simultaneamente, é necessário submeter um ficheiro Python que contém todo o código submetido no quiz. **Sem esta submissão o trabalho não poderá ser avaliado**, independentemente do resultado obtido no Moodle. Esse ficheiro deve chamar-se **LagartasSatisfeitas_IIA_25_26_alunoXXXXX.py**, em que substituem XXXXX pelo número do aluno.

### Prazo
Os testes no Moodle fecham às 23:59 de 6a feira, 5 de Dezembro.

A submissão do ficheiro Python fecha uma hora mais tarde.


## **Bom trabalho!**
<left> <img src="figures/lagarta_P2_fim.png" width="300" /> </center>

(imagens meramente ilustrativas retiradas e/ou adaptadas de: https://jayisgames.com/review/lime-rick.php)